# Exercise 1: Supervised Classification (EEG → Sleep Staging)

## Context: Sleep Staging and its Clinical Importance

**What is Sleep Staging?**

Sleep staging is the process of classifying sleep into distinct physiological states based on neurophysiological recordings, primarily electroencephalography (EEG). This classification is fundamental to understanding sleep architecture and diagnosing sleep disorders.

**The Five Sleep Stages:**

1. **Wake (W)**: Active consciousness with eyes open or closed, characterized by high-frequency, low-amplitude EEG activity (alpha and beta rhythms when relaxed).

2. **N1 (NREM Stage 1)**: Light sleep transition from wakefulness. EEG shows theta activity (4-8 Hz) and disappearance of alpha rhythms. Easily disrupted by external stimuli.

3. **N2 (NREM Stage 2)**: Consolidated light sleep accounting for ~50% of total sleep time. Distinguished by sleep spindles (11-16 Hz bursts in the sigma band) and K-complexes (sharp negative deflections followed by positive components).

4. **N3 (NREM Stage 3 / Slow-Wave Sleep)**: Deep restorative sleep dominated by high-amplitude delta waves (<4 Hz). Critical for physical restoration, immune function, and memory consolidation. Difficult to awaken from.

5. **REM (Rapid Eye Movement)**: Paradoxical sleep with high-frequency, low-amplitude EEG resembling wakefulness, but with muscle atonia and rapid eye movements. Primary stage for vivid dreaming and emotional memory processing.


The goal is to classify sleep stages (Wake, N1, N2, N3, REM) from EEG spectral features (Power Spectral Density - PSD). The sampling unit is an **epoch** (typically 30 seconds), and for each epoch we have a feature vector of powers at discrete frequencies. The challenge lies in capturing subtle spectral signatures that distinguish stages while generalizing across subjects with different age, sex, and neurophysiology.

---

## Learning Objectives

- Load and inspect EEG spectral features for multiple subjects
- Visualize stage-wise mean PSD and understand canonical EEG bands (δ, θ, α, σ, β)
- Define a preprocessing pipeline (log/dB scaling + standardization)
- Compare supervised classifiers with cross-subject validation (no leakage)
- Engineer band-level features and contrast them with PCA-based dimensionality reduction
- Train the best model on the full training set, persist it, and evaluate on unseen subjects



## Setup and libraries
We import core libraries (NumPy, Matplotlib, PyTorch for loading features saved as tensors, and tqdm for progress bars).

In [104]:
%matplotlib inline
from typing import List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import torch
from tqdm import tqdm

# preprocessing pipeline
from sklearn.model_selection import cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

from sklearn.base import clone
from sklearn.decomposition import PCA

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn import tree as tree
from sklearn.svm import SVC

from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
)

from pickle import dump, load


## Data and features
This section defines:
- The mapping of sleep stages and their colors for visualization;
- EEG frequency bands (delta, theta, alpha, sigma, beta);
- Utility functions to read per-subject features (epoch-wise PSD) and convert to dB;
- A plotting routine that shows, for each subject, the stage-wise mean PSD.

Practical notes:
- PSD are in µV²/Hz; the logarithmic (dB) scale improves interpretability across stages;
- Ensure the available `freqs` cover the band limits before drawing vertical lines;
- Inspect within-subject variability: strong differences motivate standardization.

Guiding questions:
- Which bands are most discriminative (e.g., sigma enhancements in N2/N3)?
- Do the power distributions show outliers suggesting robust scaling?
- Are class labels balanced across stages? If not, metrics like balanced accuracy are preferable.

In [105]:
# utility contants

# id corresponding to each sleep stage
ID_STAGES = {
    0: "W",
    1: "N1",
    2: "N2",
    3: "N3",
    4: "R",
}

# list of sleep stages and their corresponding colors for plotting
STAGES = ["Wake", "N1", "N2", "N3", "REM"]
STAGE_COLORS = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]

BANDS = {
    "delta": [0.5, 4.5],
    "theta": [4.5, 8.5],
    "alpha": [8.5, 11.5],
    "sigma": [11.5, 15.5],
    "beta": [15.5, 30],
}

RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)

## Exercise 1.1: Utility Functions for Data Loading and Visualization

In this section, we define three essential utility functions that form the foundation of our analysis:

1. **`read_subject_features()`**: Loads pre-computed PSD features, frequency bins, and sleep stage labels from disk for a given subject. 

2. **`power_scale()`**: Converts PSD values from linear scale (µV²/Hz) to logarithmic scale (dB). This transformation is critical for:
   - **Visualization**: Makes the wide dynamic range of power values more interpretable
   - **Numerical stability**: Reduces the impact of extreme outliers on downstream processing
   - **Feature engineering**: Many classifiers benefit from more normally-distributed features

3. **`plot_and_fetch_subjects_features()`**: Creates a comprehensive visualization showing the mean PSD spectrum for each sleep stage across multiple subjects. This function:
   - Plots stage-wise spectral profiles with EEG frequency band markers
   - Enables quick visual assessment of inter-subject variability
   - Returns the loaded data in a format ready for machine learning pipelines

**Implementation Notes:**

- All PSD data are in **µV²/Hz** (microvolts squared per Hz), the standard unit for EEG power spectral density
- The frequency bands (delta, theta, alpha, sigma, beta) are marked with vertical lines to aid interpretation
- Colors are consistent across all visualizations for easy stage identification

In [106]:
def read_subject_features(i: int) -> Tuple[np.array, np.array, np.array]:
    """
    Read features, frequencies, and stages for a given subject.
    
    Loads pre-computed power spectral density (PSD) features, frequency bins,
    and sleep stage labels from PyTorch tensor files for a specific subject.

    Parameters
    ----------
    i : int
        Subject index (0-based indexing).

    Returns
    -------
    features : np.array
        2D array of shape (n_samples, n_features) containing PSD values in µV²/Hz,
        where n_samples is the number of epochs and n_features is the number of 
        frequency bins.
    stages : np.array
        1D array of shape (n_samples,) containing sleep stage labels (0=Wake, 
        1=N1, 2=N2, 3=N3, 4=REM).
    freqs : np.array
        1D array of shape (n_features,) containing frequency values in Hz 
        corresponding to each feature column.

    Examples
    --------
    >>> features, stages, freqs = read_subject_features(0)
    >>> print(f"Subject 0: {features.shape[0]} epochs, {features.shape[1]} frequencies")
    >>> print(f"Frequency range: {freqs.min():.1f} - {freqs.max():.1f} Hz")
    """

    features = torch.load(f"data/features/subject_{i}.pt").float().numpy()
    freqs = torch.load(f"data/features/freqs_{i}.pt").float().numpy()
    stages = torch.load(f"data/stages/subject_{i}.pt").long().numpy() - 1

    return features, stages, freqs


def power_scale(x: np.array) -> np.array:
    """
    Convert power spectral density from linear to logarithmic (dB) scale.
    
    Transforms PSD values in µV²/Hz to decibel scale for improved visualization
    and numerical stability. The transformation applies floor threshold to avoid
    log of zero or negative values.

    Parameters
    ----------
    x : np.array
        Input array containing power spectral density values in µV²/Hz.
        Can be 1D or 2D (n_samples, n_features).

    Returns
    -------
    psd_db : np.array
        Power spectral density in dB scale (10 * log10(PSD)).
        Same shape as input array.

    Examples
    --------
    >>> psd_linear = np.array([1e-6, 1e-5, 1e-4])  # in µV²/Hz
    >>> psd_db = power_scale(psd_linear)
    >>> print(psd_db)  # Values in dB scale
    """

    pass


def plot_and_fetch_subjects_features(
    n_subjects: int = 10,
    ncol: int = 5,
    nrows: int = 2,
    figsize: Tuple[int, int] = (15, 6),
) -> Tuple[List[np.array], List[np.array], np.array]:
    """
    Load multiple subjects' data and visualize stage-wise mean PSD spectra.
    
    Creates a grid of subplots showing the average power spectral density for each
    sleep stage across multiple subjects. Includes frequency band markers for the
    canonical EEG bands (delta, theta, alpha, sigma, beta).

    Parameters
    ----------
    n_subjects : int, optional
        Number of subjects to load and plot. Default is 10.
    ncol : int, optional
        Number of columns in the subplot grid. Default is 5.
    nrows : int, optional
        Number of rows in the subplot grid. Default is 2.
    figsize : Tuple[int, int], optional
        Figure size as (width, height) in inches. Default is (15, 6).

    Returns
    -------
    X : List[np.array]
        List of feature arrays, one per subject. Each array has shape 
        (n_epochs, n_frequencies).
    y : List[np.array]
        List of stage label arrays, one per subject. Each array has shape (n_epochs,).
    freqs : np.array
        1D array of frequency bins in Hz (same for all subjects).

    Examples
    --------
    >>> features, stages, freqs = plot_and_fetch_subjects_features(n_subjects=5, ncol=5, nrows=1)
    >>> print(f"Loaded {len(features)} subjects")
    >>> print(f"Total epochs: {sum(len(s) for s in stages)}")
    """

    # create fig and axes with plt.subplots

    X, y = [], []

    for i in tqdm(range(n_subjects), desc="Reading data"):
        features, stages, freqs = read_subject_features(i)

        # save the data by appending to a list
        X.append(features)
        y.append(stages)

        # now let's plot the mean PSD per stage for each subject

        # Optional: draw vertical lines for each band


    return X, y, freqs


## Initial exploration and cross-subject comparison
We produce two visualizations:
1) a quick preview on 2 subjects for fast debugging;
2) an overview on 10 subjects to capture stage- and band-specific patterns.

What to look for:
- Inter-subject differences (absolute scale and spectral shape);
- Bands that are more stable or more variable;
- Potential artifacts or anomalous spectral peaks.

In [107]:
n_subjects = 2
nrows, ncol = 1, 2
figsize = (6, 4)

_, _, _ = plot_and_fetch_subjects_features(
    n_subjects=n_subjects, ncol=ncol, nrows=nrows, figsize=figsize
)

Reading data: 100%|██████████| 2/2 [00:00<00:00, 53.72it/s]


In [108]:
# Fetch the data of the first 10 subjects and plot their mean PSD per stage
n_subjects = 10
nrows, ncol = n_subjects // 5, 5
figsize = (9, 6)

features, stages, freqs = plot_and_fetch_subjects_features(
    n_subjects=n_subjects, ncol=ncol, nrows=nrows, figsize=figsize
)

Reading data: 100%|██████████| 10/10 [00:00<00:00, 80.70it/s]


## Exercise 1.2: Preprocessing Pipeline Construction

In this section, we build a preprocessing pipeline that transforms raw PSD features into a format suitable for machine learning. 

### Why Preprocessing Matters

Machine learning algorithms make implicit assumptions about data distribution and scale:
- **SVM and kNN** are sensitive to feature magnitudes (features with larger ranges dominate distance calculations)
- **Logistic Regression** converges faster with standardized features
- **Tree-based methods** (Random Forest, Decision Trees) are scale-invariant but can still benefit from log-transforms

### The Preprocessing Pipeline 

`power_scale` -> `standard_scaling`

Note:
- use  `make_pipeline` to build an sklearn-compliant machine learning pipeline
- combine `FunctionTransformer` with `power_scale` to create your custom pipeline step
- use `StandardScaler` to scale ur data before classification

Also:
`StandardScaler` will derive the runing mean and std when you run the pipeline `.fit()` method.
For each feature the scaler will compute `z = (x - μ) / σ` during the `forward` step.

In this way the pipeline will estimate the running mean and std using only the training data and then transform both train and test. Remember that this is the only way to correctly perform standardization or normalization in general!!

In [109]:
# Use make_pipeline() to chain FunctionTransformer(power_scale) and StandardScaler()
# Step 1: Convert to dB scale
# Step 2: Standardize - out: (mean=0, std=1)



## Exercise 1.3: Data Preparation and Group Labeling

### The Data Leakage Problem

When concatenating multiple subjects' data into a single dataset, we lose track of subject identity. This creates a critical risk during cross-validation: **data leakage**.

**What is data leakage in this context?**

If we perform standard k-fold cross-validation on concatenated data, epochs from the same subject could be split between training and testing sets. This allows the model to "learn" subject-specific patterns during training and exploit them during testing, leading to **artificially inflated performance estimates** that don't generalize to truly unseen subjects.

**Example of the problem:**
```
Subject A: [epoch1, epoch2, epoch3, epoch4, epoch5, epoch6]
Subject B: [epoch7, epoch8, epoch9, epoch10, epoch11, epoch12]

Standard 3-fold CV might create:
  Fold 1: Train=[epoch1,epoch2,epoch7,epoch8] Test=[epoch3,epoch4,epoch9,epoch10]
          ❌ Subject A is in both train and test!
```

### The Solution: Grouped Cross-Validation

We use **GroupKFold** (or equivalently, set `cv=n_subjects` with a `groups` parameter) to ensure that all epochs from the same subject stay together in either the training or testing set, but never both.

**Leave-One-Subject-Out (LOSO) Cross-Validation:**
- Each fold holds out all data from exactly one subject for testing
- Training uses data from all other subjects
- This simulates the realistic scenario of deploying the model on new patients

### Data Verification

After concatenation, we verify:
1. **Shapes match**: Total epochs = sum of individual subject epochs
2. **Group integrity**: Each subject ID appears the expected number of times
3. **Stage distribution**: Check for class imbalance across the full dataset

This verification step catches data loading errors and provides insight into whether specialized techniques (e.g., class weighting, stratified sampling) might be needed.

In [110]:

# Each epoch must be labeled with its subject ID
# create a group variable with subject IDs
# groups = ...
# Shape: (n_total_epochs,)


# Concatenate all subjects' data into unified arrays
# Shape: (n_total_epochs, n_frequencies)

# same for stages
# Shape: (n_total_epochs,)

# Verify the data preparation

# print :
# -- number of subjects
# -- X shape, y shape, group shape
# -- Group Ids ( must match subject ids )
# -- number of epochs per subjects
# -- stage/epoch distribution

## Exercise 1.4: Classifier Comparison with Leave-One-Subject-Out Cross-Validation

### The Complete Classification Pipeline

Now that we have prepared our data with proper grouping to prevent leakage, we can build complete classification pipelines that combine preprocessing and classification:

```
preprocessing_pipeline → classifier
```

This translates to:
```
power_scale → standardization → classifier
```

### Why Compare Multiple Classifiers?

Different classifiers make different assumptions about the data:

1. **k-Nearest Neighbors (kNN)**: Non-parametric, assumes similar inputs have similar outputs. Sensitive to feature scaling and the curse of dimensionality.

2. **Random Forest**: Ensemble of decision trees. Robust to outliers, handles non-linear relationships, provides feature importance. Less affected by feature scaling.

3. **Logistic Regression**: Linear model with probabilistic interpretation. Fast, interpretable, works well when classes are linearly separable in feature space.

4. **Support Vector Machine (SVM with RBF kernel)**: Finds optimal decision boundary in high-dimensional space. Powerful for non-linear classification but sensitive to hyperparameters.

5. **Decision Tree**: Interpretable hierarchical model. Prone to overfitting but useful as a baseline.

### Performance Metrics

We evaluate classifiers using multiple metrics to get a comprehensive performance picture:

- **Accuracy**: Overall fraction of correct predictions. Simple but can be misleading with imbalanced classes.
  
- **Balanced Accuracy**: Average recall across classes. Robust to class imbalance (important since sleep stages have different frequencies).

- **F1-weighted**: Harmonic mean of precision and recall, weighted by class support. Balances false positives and false negatives.

- **Precision-weighted**: Fraction of true positives among predicted positives, weighted by class support. Minimizes false alarms.

- **Recall-weighted**: Fraction of true positives among actual positives, weighted by class support. Minimizes missed detections.

### The `evaluate_pipelines()` Function

Below, we implement a utility function that:
1. Creates complete pipelines by combining the preprocessing pipeline with each classifier
2. Performs leave-one-subject-out cross-validation (using the `groups` parameter)
3. Computes all performance metrics for each fold
4. Returns a summary table with mean ± std for each metric
5. Identifies the best-performing classifier based on accuracy

In [111]:
scoring_metrics = [
    "accuracy",
    "f1_weighted",
    "balanced_accuracy",
    "precision_weighted",
    "recall_weighted",
]

classifiers = [
    KNeighborsClassifier(n_neighbors=5),
    RandomForestClassifier(n_estimators=100, random_state=42),
    LogisticRegression(max_iter=1000, random_state=42),
    SVC(kernel="rbf", C=1, gamma="scale", probability=True, random_state=42),
    DecisionTreeClassifier()
]

def evaluate_pipelines(
    preprocessing_pipeline,
    X : np.array,
    y : np.array,
    groups : np.array,
    n_subjects : int,
    classifier : List = classifiers,
    metrics : List[str] = scoring_metrics,
) -> pd.DataFrame:
    """
    Evaluate multiple classification pipelines using grouped cross-validation.
    
    Performs leave-one-subject-out cross-validation to assess classifier performance
    while preventing data leakage across subjects. Each fold trains on n-1 subjects
    and tests on the held-out subject. Reports multiple performance metrics and
    identifies the best-performing classifier.

    Parameters
    ----------
    preprocessing_pipeline : sklearn.pipeline.Pipeline
        The preprocessing pipeline to apply before classification (e.g., scaling,
        feature transformation).
    X : np.array
        Feature matrix of shape (n_samples, n_features) containing concatenated
        data from all subjects.
    y : np.array
        Target labels of shape (n_samples,) with sleep stage annotations.
    groups : np.array
        Group labels of shape (n_samples,) indicating subject membership for each
        sample. Used to ensure cross-validation respects subject boundaries.
    n_subjects : int
        Total number of unique subjects in the dataset. Must equal the number of
        unique values in `groups`.
    classifier : List, optional
        List of sklearn classifier instances to evaluate. Default is the global
        `classifiers` list.
    metrics : List[str], optional
        List of metric names to compute during cross-validation. Default is the
        global `scoring_metrics` list.

    Returns
    -------
    results_df : pd.DataFrame
        DataFrame with classifiers as index and metrics as columns. Each cell
        contains the mean ± std of the metric across folds, formatted as percentage.

    Examples
    --------
    >>> from sklearn.preprocessing import StandardScaler
    >>> from sklearn.pipeline import make_pipeline
    >>> preprocessing = make_pipeline(StandardScaler())
    >>> results = evaluate_pipelines(preprocessing, X_train, y_train, groups, n_subjects=10)
    >>> print(results)
    """

    # assert that the number of distinct groups is equal to n_subjects
    assert len(np.unique(groups)) == n_subjects, "Number of unique groups must equal n_subjects"

    # perform cross-validation for each classifier pipeline

    # construct the classification pipelines
    pipelines = []
    
    table_rows = []
    for i, pipe in tqdm(enumerate(pipelines), total=len(pipelines), desc="Evaluating pipelines"):

        # use cross_validate to perform cross validation
        # set cv = n_subjects and groups = groups to perform LOSO
        # use the passed metrics as scoring in the cross_validate method
        pass

    # print the best classifier        

    return pd.DataFrame(table_rows).set_index("Classifier")


In [112]:
# results_df = evaluate_pipelines( ... )
# results_df - uncomment this to visualize the dataframe once builded


## Exercise 1.5: Feature Engineering - Domain Knowledge vs Data-Driven Approaches

### The Dimensionality Challenge

Our current feature space contains hundreds of frequency bins (one PSD value per frequency from 0.5 to 30 Hz). While this captures fine-grained spectral information, it presents several challenges:

1. **Curse of dimensionality**: High-dimensional spaces are sparse, making distance-based methods (kNN, SVM) less effective
2. **Computational cost**: More features mean longer training times and higher memory requirements
3. **Overfitting risk**: Many features relative to samples can lead to models that memorize noise
4. **Interpretability**: Hundreds of coefficients are difficult to understand clinically

### Two Feature Engineering Strategies

We compare two fundamentally different approaches to dimensionality reduction:

#### Strategy 1: Domain-Knowledge Features (Band Averaging)

**Concept**: Leverage neuroscience knowledge by computing mean power within canonical EEG frequency bands.

**The Five EEG Bands:**
- **Delta (0.5-4.5 Hz)**: Deep sleep marker, high in N3
- **Theta (4.5-8.5 Hz)**: Drowsiness and light sleep, prominent in N1
- **Alpha (8.5-11.5 Hz)**: Relaxed wakefulness with closed eyes
- **Sigma (11.5-15.5 Hz)**: Sleep spindles characteristic of N2
- **Beta (15.5-30 Hz)**: Active thinking and REM sleep

**Pipeline**: `power_scale → extract_band_features → standardization → classifier`

**Advantages:**
- **Interpretability**: Each feature has clear physiological meaning
- **Dimensionality reduction**: ~500 frequencies → 5 band powers (100× reduction)
- **Robustness**: Averaging reduces sensitivity to frequency-specific noise
- **Clinical relevance**: Aligns with how sleep experts analyze PSG data

**Disadvantages:**
- **Information loss**: Fine-grained spectral details within bands are discarded
- **Fixed bands**: Assumes traditional band boundaries are optimal (may vary by age, individual)

#### Strategy 2: Data-Driven Features (PCA)

**Concept**: Let the data determine the most important patterns through unsupervised learning.

**Pipeline**: `power_scale → standardization → PCA(n_components=5) → classifier`

**How PCA works:**
1. Centers the data (zero mean)
2. Finds directions of maximum variance (principal components)
3. Projects data onto top-k components
4. Each component is a weighted combination of all original frequencies

**Advantages:**
- **Data-driven**: Discovers patterns specific to this dataset
- **Variance maximization**: Retains maximum information with fewer dimensions
- **Flexibility**: Can easily adjust number of components (2, 5, 16, etc.)
- **Decorrelation**: Components are orthogonal, removing redundancy

**Disadvantages:**
- **Black box**: Components are linear combinations without clear physiological meaning
- **Dataset-specific**: Optimal components may not transfer to other populations
- **Requires standardization**: Sensitive to feature scaling

### Comparison Strategy

We evaluate both approaches using:
- **Same preprocessing**: Both apply `power_scale` before feature extraction
- **Same classifiers**: All 5 classifiers tested on both feature sets
- **Same CV splits**: Identical leave-one-subject-out folds ensure fair comparison
- **Same metrics**: All 5 performance metrics computed

This allows us to isolate the impact of feature engineering from other pipeline choices.

In [113]:
def extract_band_features(x: np.array) -> np.array:
    """
    Extract average power within predefined EEG frequency bands.
    
    Computes the mean PSD within each canonical EEG band (delta, theta, alpha,
    sigma, beta) to create a compact, interpretable feature representation.
    This reduces dimensionality from hundreds of frequency bins to 5 band powers.

    Parameters
    ----------
    x : np.array
        Input data of shape (n_samples, n_features) containing PSD values across
        frequencies. Should be in dB scale for optimal performance.

    Returns
    -------
    band_features : np.array
        Extracted band-averaged features of shape (n_samples, n_bands), where
        n_bands is the number of frequency bands with data coverage (typically 5).

    Examples
    --------
    >>> psd_db = power_scale(raw_psd)  # Convert to dB first
    >>> band_feats = extract_band_features(psd_db)
    >>> print(band_feats.shape)  # (n_epochs, 5) for delta, theta, alpha, sigma, beta
    """
    pass


# build preprocessing pipeline for mean band features and for pca

# preprocessing_pipeline_mean = make_pipeline( ... )

# preprocessing_pipeline_pca = make_pipeline( ... )


In [114]:
# evaluate mean preprocessing pipeline with 
# evaluate_pipeline( ... )

In [115]:
# evaluate pca preprocessing pipeline with 
# evaluate_pipeline( ... )

## Exercise 1.6: PCA Hyperparameter Tuning - Finding the Optimal Number of Components

### The Hyperparameter Selection Problem

In Exercise 1.5, we compared band averaging (5 features) against PCA with 5 components. But **how do we know that 5 is the optimal number of components?**

This is a critical hyperparameter that affects:
- **Model capacity**: Too few components → underfitting (missing important patterns)
- **Generalization**: Too many components → overfitting (capturing noise)
- **Computational efficiency**: More components mean longer training times
- **Interpretability**: Fewer components are easier to analyze

### Two Complementary Perspectives

We'll evaluate PCA's optimal dimensionality through two lenses:

#### Perspective 1: Variance Explained (Unsupervised)

**Question**: How much of the original data's variance do we retain?

**Method**: Fit PCA on the full dataset and compute the cumulative explained variance ratio.

**Formula**: 
$$\text{Explained Variance Ratio} = \frac{\sum_{i=1}^{k} \lambda_i}{\sum_{i=1}^{n} \lambda_i}$$

where $\lambda_i$ are the eigenvalues (variance along each principal component).

**What to look for:**
- **Elbow point**: Where adding more components yields diminishing returns
- **Target threshold**: Often aim for 90-95% variance retention
- **Interpretation**: Shows information preservation, not predictive power

**Limitation**: High explained variance doesn't guarantee good classification! Variance and class discriminability are different objectives.

#### Perspective 2: Classification Performance (Supervised)

**Question**: Which dimensionality maximizes predictive accuracy on held-out subjects?

**Method**: For each value of `n_components` [2, 4, 8, 16, 32, 64, 128, 256]:
1. Build pipeline: `power_scale → standardization → PCA(n_components) → classifier`
2. Perform leave-one-subject-out cross-validation
3. Compute mean accuracy ± standard deviation across folds
4. Plot accuracy curves for each classifier

**What to look for:**
- **Performance plateau**: Where accuracy stops improving (optimal region)
- **Overfitting zone**: Where performance degrades with more components
- **Classifier differences**: Some models (linear) may need fewer components than others (non-linear)

**Why this matters**: This directly measures generalization to unseen subjects—our true objective.

### Implementation Strategy

Below, we:
1. **Test logarithmically-spaced values**: [2, 4, 8, 16, 32, 64, 128, 256] to efficiently cover the range
2. **Visualize explained variance**: Understand how PCA compresses information
3. **Compare all classifiers**: Each may have different optimal dimensionalities
4. **Use error bars**: Standard deviation across folds shows stability

### Expected Insights

- **Low dimensions (2-8)**: May work well if classes are linearly separable in PCA space
- **Medium dimensions (16-64)**: Often optimal—balances capacity and generalization
- **High dimensions (128-256)**: Risk overfitting unless using regularized classifiers (e.g., SVM, Logistic Regression with L2)
- **Classifier-specific patterns**: Linear models plateau early; non-linear models may benefit from more components

In [116]:
# first define some components and plot the explained variance of the PCA
n_components = [ 2, 4, 8, 16, 32, 64, 128, 256 ]

explained_variance_results = []

# manually perform the power-scaling and standardization
# X_pow = ...
# X_scaled = ...

for n_component in tqdm( n_components, desc="Evaluating PCA components"):
    
    # fit pca with X_scaled

    # get the explained variance as sum of the ratios `pca.explained_variance_ratio_`
    pass

# Plot explained variance vs number of components


Evaluating PCA components: 100%|██████████| 8/8 [00:00<00:00, 180400.17it/s]


In [117]:

# Plot accuracy vs number of PCA components with error bars

# fig, axes = plt.subplots( ... )
axes = []

for i, ax in enumerate(axes):
    classifier = classifiers[i]
    
    print(f"\nEvaluating classifier: {classifier.__class__.__name__}")
    # create the pipelines for each pca components
    # pipelines = ...

    # perform cross-validation for each classifier pipeline using cross_validate
    cv_results = []
    
    # ...

    cv_df = pd.DataFrame(cv_results)
    ax = axes[i]
    ax.errorbar(
        cv_df['n_components'],
        cv_df['mean_accuracy'],
        yerr=cv_df['std_accuracy'],
        marker='o',
        linewidth=2,
        capsize=5,
    )
    ax.set_xlabel('Number of PCA Components')
    ax.set_ylabel('Cross-Validated Accuracy')
    ax.set_title(f'Classifier: {classifier.__class__.__name__}')
    ax.grid(True, alpha=0.3)
    ax.set_xscale('log', base=2)

plt.tight_layout()
plt.show()

# print best classifier

<Figure size 640x480 with 0 Axes>

## Exercise 1.7: Model Deployment - Training, Persistence, and Evaluation on Unseen Data

### From Cross-Validation to Production

Up to this point, we've used cross-validation to **estimate** generalization performance. Now we transition to **deploying** a final model for real-world use.

### The Deployment Pipeline
- Choose the Best Pipeline
- Train on Full Training Set ( eventho we could use LOSO cross validation results to avoid specific subjects )
- Save the model to the disk to restore it in any moment
- Load the saved model and test it to new and unseen data

Notes:
We serialize the trained pipeline to disk using Python's `pickle` module:

```python
with open("best_sklearn_model.pkl", "wb") as f:
    dump(best_pipeline, f)
```

**Benefits of persistence:**
- **Reproducibility**: Same model can be reused without retraining
- **Deployment**: Load model in production environments (web services, embedded systems)
- **Versioning**: Save different model versions for comparison
- **Efficiency**: Avoid expensive retraining for inference


We evaluate on subjects 11-15 (completely unseen during training or CV):

**Metrics:**
1. **Confusion Matrix (normalized)**: Shows per-class accuracy and common misclassifications
   - Diagonal = correct predictions
   - Off-diagonal = confusion patterns (e.g., N1 ↔ N2, Wake ↔ REM)
   
2. **Classification Report**: Provides per-class precision, recall, F1-score
   - **Precision**: Of all predicted N2, how many were actually N2?
   - **Recall**: Of all actual N2, how many did we detect?
   - **F1-score**: Harmonic mean balancing precision and recall

### Interpreting Results

**What to look for in the confusion matrix:**
- **Strong diagonal**: High per-stage accuracy
- **Adjacent confusions**: N1 ↔ N2 confusion is common (transitional stages)
- **Wake vs REM**: May confuse due to similar high-frequency EEG
- **N3 accuracy**: Should be high (distinctive slow-wave activity)

**What to look for in the classification report:**
- **Macro vs weighted averages**: 
  - Macro = simple average across classes (treats all stages equally)
  - Weighted = accounts for class imbalance (more realistic for this application)
- **Per-class performance**: Identify which stages are hardest to classify
- **Comparison to CV**: Test accuracy should be similar to CV estimates (validates our methodology)

In [118]:
# now deploy the best model on the whole training set and save it

# best_classifier = 
# n_components_best = 

# best_pipeline = make_pipeline( ... )

# save the model
# with open("best_sklearn_model.pkl", "wb") as f:
#    dump(best_pipeline, f)

# load the model
# with open("best_sklearn_model.pkl", "rb") as f:
#    loaded_model = load(f)

# evaluate the loaded model on subjects 11 to 15

y_pred_all = []
y_true_all = []


# plot the confusion matrix with ConfusionMatrixDisplay.from_predictions 


# plot the classification report as a dataframe ( output_dict=True in classification_report )
